# Human-in-the-Loop (HITL) — Simple Refund Approval

This example demonstrates how **Human-in-the-Loop (HITL)** works with the OpenAI Agents SDK.

The Customer Support Agent can use an `issue_refund` tool.  
Because the tool is configured with `needs_approval=True`, the tool **cannot execute until a human approves it**.

## HITL Workflow

**1. User requests a refund**  
↓  
**2. Agent selects the `issue_refund` tool**  
↓  
**3. `needs_approval=True` triggers HITL**  
↓  
**4. Agent execution pauses**  
↓  
**5. Check `result.interruptions`**  
↓  
**6. Save the paused workflow using `result.to_state()`**  
↓  
**7. Human reviews the proposed action**  
↓  
**Approve → `state.approve()`**  
or  
**Reject → `state.reject()`**  
↓  
**8. Resume the workflow using `Runner.run(agent, state)`**  
↓  
**9. Agent completes the workflow**

> **Key Idea:** The AI agent proposes the action, but the human controls whether the action is allowed to execute.

In [2]:
import os
from dotenv import load_dotenv

from agents import Agent, Runner
from agents.decorators import tool

In [3]:
load_dotenv()

True

## Define the Human Approval Policy

Not every refund needs human approval.

The following function defines our approval rule:

- Return `False` → Human approval is NOT required
- Return `True` → Human approval IS required

In this example, only refunds over $100 require approval.

## Create the Refund Tool

The `issue_refund` tool represents the action the Customer Support Agent can perform to process a refund.

- `@tool` converts the Python function into an agent tool.
- `needs_approval=True` requires **human approval every time** the tool is called.
- The tool will **not execute immediately** when selected by the agent.
- Instead, the workflow pauses and creates an **interruption**.
- After human approval, the workflow resumes and the refund tool executes.

> **Note:** This is a simulated refund for demonstration purposes. No real payment transaction is performed.


In [4]:
@tool(needs_approval=True)
async def issue_refund(
    order_id: str,
    amount: float
) -> str:
    """Process a simulated customer refund."""

    return (
        f"Refund of ${amount:.2f} "
        f"processed for order {order_id}."
    )


## Step 4 — Create the Customer Support Agent

The agent receives customer requests and decides when to use the
`issue_refund` tool.

The agent does not implement the HITL decision itself.

The tool's `needs_approval` policy determines whether execution
must pause for human approval.

In [5]:
agent = Agent(
    name="Support Agent",

    instructions="""
    You are a customer support agent.

    When a customer asks for a refund,
    you MUST use the issue_refund tool.

    Do not claim that a refund was processed
    without using the tool.
    """,

    tools=[issue_refund],

    model="gpt-5-mini"
)

## Test the  Process

In [6]:
result = await Runner.run(
    agent,
    "Use the issue_refund tool to refund $175 for order ORD-101."
)

In [7]:
state = result.to_state()
interruption = result.interruptions[0]

print("Tool:", interruption.name)
print("Arguments:", interruption.arguments)

Tool: issue_refund
Arguments: {"order_id":"ORD-101","amount":175}


In [8]:
state = result.to_state()

interruption = result.interruptions[0]

decision = input("Approve refund? (yes/no): ")

if decision.strip().lower() == "yes":
    state.approve(interruption)
    print("Human Decision: APPROVED")
else:
    reason = input("Enter rejection reason: ")

    state.reject(
        interruption,
        rejection_message=reason
    )

    print("Human Decision: REJECTED")

Human Decision: APPROVED
